# IIT - Introduction à PyPhi et Integrated Information Theory

## Navigation
**Série** : [IIT](./README.md) | **Partie** : Fondements
**Précédent** : -- | **Suivant** : --

## Objectifs d'apprentissage
1. **Comprendre** les principes de l'Integrated Information Theory (IIT)
2. **Manipuler** les objets PyPhi : Network, Subsystem, CES
3. **Calculer** et interpréter la valeur Φ (Phi) d'un système
4. **Explorer** la structure causale (Cause-Effect Structure)

### Prérequis
- Python 3.7+ (environnement Conda recommandé)
- Notions de base en théorie de l'information
- Connaissances en algèbre linéaire (matrices de transition)

### Durée estimée : 60 minutes

---

### **Sommaire**
1. [Introduction à l'IIT](#sec1)  
2. [Installation et importation de PyPhi](#sec2)  
3. [Création d'un réseau simple (Network)](#sec3)  
4. [Définition d'un Subsystem et calcul de \(\Phi\)](#sec4)  
5. [Concepts, mécanismes, et structure cause-effet (CES)](#sec5)  
6. [Exploration d'exemples avancés (macro, actual causation…)](#sec6)  
7. [Ressources complémentaires](#sec7)

---

<a id="sec1"></a>
## 1. Introduction à l'IIT

**Objectif** : l'**Integrated Information Theory** (IIT), introduite par Giulio Tononi et collaborateurs, propose une mesure de la conscience à partir de la quantité d'information intégrée par un système. En pratique :

- Un **système** (réseau de neurones, circuit logique, etc.) est représenté par une **Transition Probability Matrix (TPM)**.
- L'information intégrée est notée \(\Phi\) (Phi). PyPhi calcule \(\Phi\) pour des sous-systèmes et explore la structure causale (Cause-Effect Structure, ou *CES*).
- L'**objectif pédagogique** de ce notebook est de créer un petit réseau, de définir un état, et de calculer \(\Phi\) pour différents sous-ensembles (subsystèmes).

### Objectifs du TP
1. Comprendre la représentation d'un système dynamique discret via une TPM.
2. Manipuler les objets de base de PyPhi : `Network`, `Subsystem`.
3. Calculer et interpréter la valeur \(\Phi\) (degré d'irréductibilité).
4. Explorer la structure causale (CES) d'un état donné.

### Résumé des concepts clés

| Concept | Explication simple | Analogie |
| :--- | :--- | :--- |
| **TPM** | Les règles du jeu. Comment le système évolue. | Les lois de la physique. |
| **État** | La configuration actuelle (0 ou 1) des nœuds. | Une photo instantanée du cerveau. |
| **\(\Phi\) (Big Phi)** | Le niveau d'intégration du système entier. | La solidité d'un nœud marin (tient-il tout seul ?). |
| **MIP** | Le point faible du système (Minimum Information Partition). | Le maillon faible d'une chaîne. |
| **CES** | La forme géométrique de l'information intégrée. | La "forme" d'une pensée. |

---

<a id="sec2"></a>
## 2. Installation et importation de PyPhi

> **Remarque** : PyPhi exige parfois une installation depuis la source ou via `pip`.  
> **Pré-requis** : Python 3.7+ (recommandé).

⚠️ **Attention aux versions** : PyPhi est une librairie scientifique complexe qui dépend souvent de versions spécifiques de bibliothèques (numpy, pandas). Si vous rencontrez des erreurs d'importation ou de compatibilité `numpy`, c'est souvent dû à une version de Python trop récente (3.10+).

Il est vivement conseillé d'utiliser un environnement Conda dédié :
```bash
conda create --name pyphi python=3.7 -y
conda activate pyphi
pip install pyphi
```

Sélectionner l'environnement nouvellement créé comme noyau

In [1]:
# Dans un terminal ou dans un environnement conda :
! pip install pyphi


Defaulting to user installation because normal site-packages is not writeable


Une fois PyPhi installe, importons la librairie et verifions la version disponible.

In [2]:
import pyphi

print("PyPhi version:", pyphi.__version__)

ImportError: cannot import name 'Iterable' from 'collections' (C:\Users\jsboi\AppData\Local\Programs\Python\Python313\Lib\collections\__init__.py)

<a id="sec3"></a>
## 3. Création d’un réseau simple (Network)

### 3.1. Structure d’un réseau

Un **Network** dans PyPhi est défini par :
- Une **TPM** (`ExplicitTPM`) en forme de matrice multidimensionnelle ou un array 2D.
- Une **connectivity matrix** (CM), indiquant comment les nœuds sont connectés.

### 3.2. Exemple : un circuit logique

Nous allons créer un réseau (circuit) minimaliste à 3 nœuds (A, B, C) :

- **A** : passerelle logique, reçoit des entrées de B et C  
- **B** : un XOR des entrées de A et C  
- **C** : identique à B ou un AND/OR, selon l’exemple souhaité 

In [3]:
import numpy as np
import pyphi

# Exemple d'un petit réseau XOR
# 3 noeuds : A, B, C
# On utilise un exemple déjà inclus dans pyphi.examples

network = pyphi.examples.xor_network()

print("Nodes:", network.node_labels)
print("TPM shape:", network.tpm.shape)
print("Connectivity matrix:\n", network.cm)

ImportError: cannot import name 'Iterable' from 'collections' (C:\Users\jsboi\AppData\Local\Programs\Python\Python313\Lib\collections\__init__.py)

**Comment ça marche ?**  
- `pyphi.examples.xor_network()` renvoie un réseau (Network) de taille 3, où chaque nœud est un XOR à deux entrées (sans auto-connexion).  
- La **TPM** est stockée dans `network.tpm`, et chaque dimension correspond à un état binaire (`0/1`). La `TPM` est un tableau multidimensionnel de taille (2, 2, 2, 3), reflétant les états binaires pour 3 nœuds.
La matrice de connectivité montre les liens (XOR signifie que chaque nœud dépend des deux autres sans boucles sur soi-même).


---

### 3.3. Inspection de la TPM

Pour mieux comprendre le fonctionnement du réseau, vous pouvez examiner directement les valeurs de la matrice de transition (TPM). Celle-ci peut être soit en _state-by-state_ (SBS) ou _state-by-node_ (SBN) selon votre cas. PyPhi propose des utilitaires de conversion.


In [4]:
import pyphi.convert

sbn_tpm = network.tpm
print("TPM (state-by-node) shape:", sbn_tpm.shape)

# Pour visualiser un 'slice' ou un 'flatten' :
print("Premier 'slice' de la TPM (correspondant à un sous-état) :\n", sbn_tpm[0])

# Passer au format state-by-state pour mieux le parcourir
sbs_tpm = pyphi.convert.state_by_node2state_by_state(sbn_tpm)
print("\nTPM (state-by-state) shape:", sbs_tpm.shape)
print("Quelques lignes du TPM:\n", sbs_tpm[:4])


ImportError: cannot import name 'Iterable' from 'collections' (C:\Users\jsboi\AppData\Local\Programs\Python\Python313\Lib\collections\__init__.py)

<a id="sec4"></a>
## 4. Définition d’un Subsystem et calcul de \(\Phi\)

### 4.1. Sous-système (Subsystem)

Un **Subsystem** se définit en spécifiant :
1. Un **Network** ;
2. Un **état** du réseau (ex. `(0, 0, 0)` pour `A=0, B=0, C=0`) ;
3. Les indices de nœuds internes qu’on considère comme faisant partie du sous-système.

Ensuite, PyPhi peut calculer la valeur de \(\Phi\) pour ce sous-système.

In [5]:
# État de départ
state = (0, 0, 0)

# Créons un Subsystem pour l’ensemble des nœuds (0, 1, 2)
subsystem = pyphi.Subsystem(network, state, (0, 1, 2))

# Calcul de Phi
phi_value = pyphi.compute.phi(subsystem)

print("Φ pour le sous-système (0,1,2) à l’état (0,0,0) :", phi_value)

NameError: name 'pyphi' is not defined

### 4.2. States inaccessibles et validation (`StateUnreachableError`)

C'est une erreur fréquente rencontrée dans PyPhi.

**Le problème** : Certains états du réseau peuvent être impossibles à atteindre selon la dynamique du système (la TPM). Si on force le système dans cet état "impossible", l'IIT considère que le passé est indéfini.

**Solutions possibles** :
1. Choisir un état qui est atteignable (un état stable ou faisant partie d'un cycle).
2. Ou désactiver la validation (pour l'exercice) : `pyphi.config.VALIDATE_SUBSYSTEM_STATES = False`.

PyPhi, par défaut, peut émettre des erreurs (`StateUnreachableError`) si l'état que vous définissez n'est pas accessible depuis la TPM. Ci-dessous, on montre comment désactiver temporairement la validation pour s'assurer que PyPhi ne bloque pas le calcul.

In [6]:
# Configurer PyPhi pour ignorer les inaccessibilités d’états
pyphi.config.VALIDATE_SUBSYSTEM_STATES = False

# Réessayer un état arbitraire
test_state = (1,0,0)
subsys_test = pyphi.Subsystem(network, test_state, (0,1,2))
try:
    val_test = pyphi.compute.phi(subsys_test)
    print(f"État {test_state} -> Φ = {val_test:.5f}")
except pyphi.exceptions.StateUnreachableError:
    print("État inaccessible, PyPhi a lancé une exception !")


NameError: name 'pyphi' is not defined

On peut également calculer \\(\Phi\\) pour d'autres états accessibles:

In [7]:
# Exemple d'états stables pour éviter StateUnreachableError
for s in [(0,0,0), (0,1,1), (1,1,0)]:
    subsys = pyphi.Subsystem(network, s, (0,1,2))
    val = pyphi.compute.phi(subsys)
    print(f"État {s} -> Φ = {val:.5f}")


NameError: name 'pyphi' is not defined

<a id="sec5"></a>
## 5. Concepts, mécanismes, et structure cause-effet (CES)

PyPhi peut également détailler la structure de cause-effet (CES) :

- Chaque *concept* correspond à un **mécanisme** (ex. nœud simple ou sous-groupe de nœuds) et à son purview (cause et/ou effet).

In [8]:
ces = pyphi.compute.ces(subsystem)
print("Nombre de concepts dans la CES :", len(ces))
print("Liste des mécanismes :", ces.labeled_mechanisms)

NameError: name 'pyphi' is not defined

### 5.1. Analyse d’un concept spécifique

Chaque concept est défini par un mécanisme (un ensemble de nœuds) et un purview (un ensemble de nœuds qui subissent ses effets ou le causent). Explorons le premier concept dans la CES, pour comprendre sa cause / effet et son petit phi (\\(\\varphi\\)).


In [9]:
# Prenons le premier concept dans la CES
first_concept = ces[0]
print("Mécanisme:", first_concept.mechanism)
print("Cause purview:", first_concept.cause.purview)
print("Cause repertoire:\n", first_concept.cause.repertoire)
print("Effect purview:", first_concept.effect.purview)
print("Effect repertoire:\n", first_concept.effect.repertoire)
print("Petit phi (φ) =", first_concept.phi)


NameError: name 'ces' is not defined

### Interpretation : structure causale du premier concept

Le mecanisme {A, B} (noeuds 0 et 1) produit un concept avec :

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| **Cause purview** | (A, B, C) | Les 3 noeuds contribuent a *causer* l'etat actuel de {A, B} |
| **Effect purview** | (C,) | Le mecanisme {A, B} n'a d'effet irreductible que sur le noeud C |
| **Petit phi** | 0.5 | L'information integree par ce mecanisme specifique |

Le **repertoire de cause** montre que l'etat passe le plus probable etait (0,0,0) ou (1,1,1) avec probabilite 0.5 chacun. Le **repertoire d'effet** indique que C passera a l'etat 0 avec certitude (probabilite 1.0), ce qui est coherent avec le XOR : si A=0 et B=0, alors C = A XOR B = 0.

### Le Calcul de \(\Phi\) : Définition Intuitive

\(\Phi\) mesure à quel point le système est "plus que la somme de ses parties".

**Le calcul** : PyPhi essaie de couper le système en deux (une "bipartition") de toutes les manières possibles (A vs BC, B vs AC, etc.).
- La coupe qui fait perdre le *moins* d'information est appelée la **Minimum Information Partition (MIP)**.
- \(\Phi\) est la quantité d'information perdue au niveau de cette coupe la plus faible.
- Si \(\Phi > 0\), le système est irréductible (conscient selon l'IIT).

---

<a id="sec6"></a>
## 6. Exemple avancé : analyse de causalité et macro-subsystems

### 6.1 Actual Causation

PyPhi offre `pyphi.actual` pour analyser la causalité effective ("What caused what?"). On définit un `Transition` : état avant, état après, et on cherche les liens causaux irréductibles.

Cette extension de la théorie permet de comprendre non plus "combien de conscience" (système) mais "qui a causé quoi" (causalité événementielle). C'est utile pour analyser des séquences temporelles spécifiques.

### Interpretation : liens causaux de la transition (1,1,1) vers (0,0,0)

L'analyse revele **10 liens causaux irreductibles** pour cette transition, tous avec une force alpha = 1.0 (sauf le lien global a 2.0). On distingue :

- **Causes** (fleches vers la gauche) : chaque noeud individuel (A, B, C) est une cause irreductible pour les deux autres noeuds. Les paires {A,B}, {A,C}, {B,C} causent l'ensemble {A,B,C}.
- **Effets** (fleches vers la droite) : de maniere symetrique, chaque paire produit un effet sur le noeud restant.
- Le lien **{A,B,C} vers {A,B,C}** avec alpha = 2.0 represente l'effet global integre du systeme sur lui-meme.

Cette symetrie est typique du reseau XOR : chaque noeud depend des deux autres de maniere equivalente, ce qui produit une structure causale riche et fortement integree.

In [10]:
import pyphi
import pyphi.actual

pyphi.config.VALIDATE_SUBSYSTEM_STATES = False

before_state = (1,1,1)
after_state  = (0,0,0)

transition = pyphi.actual.Transition(
    network,
    before_state,
    after_state,
    cause_indices=(0,1,2),
    effect_indices=(0,1,2)
)

account = pyphi.actual.account(transition)
print("Nombre de causal links :", len(account))
for link in account:
    print(link)


ImportError: cannot import name 'Iterable' from 'collections' (C:\Users\jsboi\AppData\Local\Programs\Python\Python313\Lib\collections\__init__.py)

### 6.2 Macro-subsystems (coarse-graining, blackboxing)

Le module `pyphi.macro` permet d'agréger des nœuds (coarse-grain) ou de blackboxer certains éléments. 
On peut ainsi recalculer \\(\Phi\\) à une échelle différente, ce qui rejoint l’idée de l’ICT qu’on peut analyser l’information intégrée à divers niveaux.



---

## Exercice : Exploration de la structure cause-effet

**Objectifs** :
1. Analyser l'impact de la topologie du réseau sur la valeur de Φ
2. Explorer la relation entre mécanismes et purviews
3. Comparer différents réseaux pour comprendre l'intégration

**Contexte** : Vous avez vu comment calculer Φ pour un réseau XOR simple. L'IIT suggère que la valeur de Φ dépend de la façon dont l'information est intégrée dans le système. Dans cet exercice, vous allez explorer cette relation.

**Questions** :

1. **TODO** : Créez un réseau AND (où chaque nœud fait un AND de ses entrées) et comparez sa valeur Φ avec celle du réseau XOR. Quelle topologie produit plus d'intégration ?

2. **TODO** : Pour le réseau XOR, analysez comment Φ change selon l'état du système. Existe-t-il des états où Φ = 0 ?

3. **TODO** : Explorez la CES d'un sous-système à 2 nœuds vs le système complet. Comment le nombre de concepts évolue-t-il ?

In [11]:
# TODO: Exercice - Exploration de la structure cause-effet
# 1. Créer un réseau AND et comparer Φ avec XOR
# 2. Analyser Φ selon l'état
# 3. Comparer CES 2-noeuds vs 3-noeuds

# Exemple de structure pour le réseau AND
# and_tpm = ...  # Définir la TPM pour un réseau AND
# and_network = pyphi.Network(and_tpm, connectivity_matrix=cm)
# and_subsystem = pyphi.Subsystem(and_network, (0,0,0), (0,1,2))
# print("Φ (AND):", pyphi.compute.phi(and_subsystem))

pass  # Supprimer et implémenter

<a id="sec7"></a>
## 7. Ressources complémentaires

- [Documentation PyPhi](https://pyphi.readthedocs.io/en/stable/)",
- [\"Major Complex\", \"Actual Causation\"](https://github.com/wmayner/pyphi/tree/master/examples)",
- Travaux de Tononi, Koch, Oizumi, Albantakis, etc. (cf. publications IIT)",
- Pour aller plus loin : l’ICT, l’analyse fractale de l’information, etc.

**Fin du notebook**. 
Merci pour votre attention !